# Thumbnail Generation Notebook

## Parameters

In [12]:
# --- Parameters ---

MAX_WIDTH         = 150        # int or None — downscale if image width exceeds this
MAX_HEIGHT        = None        # int or None — downscale if image height exceeds this
COMPRESSION_QUALITY = 70       # 0–100 (WebP quality)
INPUT_DIRECTORY   = "full"        # relative to this notebook
OUTPUT_DIRECTORY  = "tiny"   # relative to this notebook
INCLUDE_ALPHA     = False      # keep transparency channel (RGBA) if True

## Generate Thumbnails

In [13]:
import os
from pathlib import Path
from PIL import Image

SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tiff", ".tif", ".webp"}

notebook_dir = Path().resolve()
input_dir    = (notebook_dir / INPUT_DIRECTORY).resolve()
output_dir   = (notebook_dir / OUTPUT_DIRECTORY).resolve()
output_dir.mkdir(parents=True, exist_ok=True)

def compute_size(original: tuple[int, int]) -> tuple[int, int]:
    """Return scaled (width, height) respecting MAX_WIDTH / MAX_HEIGHT, maintaining aspect ratio."""
    w, h = original
    if MAX_WIDTH is not None and w > MAX_WIDTH:
        h = round(h * MAX_WIDTH / w)
        w = MAX_WIDTH
    if MAX_HEIGHT is not None and h > MAX_HEIGHT:
        w = round(w * MAX_HEIGHT / h)
        h = MAX_HEIGHT
    return w, h

results = []  # [(input_path, output_path, input_bytes, output_bytes)]

input_files = sorted(
    p for p in input_dir.iterdir()
    if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS
)

if not input_files:
    print(f"No supported image files found in: {input_dir}")
else:
    for src in input_files:
        dst = output_dir / (src.stem + ".webp")

        with Image.open(src) as img:
            # Resolve colour mode
            if INCLUDE_ALPHA:
                img = img.convert("RGBA")
            else:
                img = img.convert("RGB")

            new_size = compute_size(img.size)
            if new_size != img.size:
                img = img.resize(new_size, Image.LANCZOS)

            img.save(dst, format="WEBP", quality=COMPRESSION_QUALITY)

        input_bytes  = src.stat().st_size
        output_bytes = dst.stat().st_size
        results.append((src, dst, input_bytes, output_bytes))
        print(f"  {src.name[:30].ljust(30)}  →  {dst.name[:30].ljust(30)}  ({new_size[0]}×{new_size[1]})")

print(f"\nDone — {len(results)} image(s) written to: {output_dir}")

  (Ringworld, Bk 1) Niven, Larry  →  (Ringworld, Bk 1) Niven, Larry  (150×247)
  (Ringworld, Bk 2) Niven, Larry  →  (Ringworld, Bk 2) Niven, Larry  (150×252)
  (Ringworld, Bk 3) Niven, Larry  →  (Ringworld, Bk 3) Niven, Larry  (150×247)
  (Ringworld, Bk 4) Niven, Larry  →  (Ringworld, Bk 4) Niven, Larry  (150×254)
  A_Deepness_in_the_Sky_-_Vernor  →  A_Deepness_in_the_Sky_-_Vernor  (150×242)
  A_Fire_Upon_the_Deep_-_Vernor_  →  A_Fire_Upon_the_Deep_-_Vernor_  (150×246)
  Bobiverse_1_We_Are_Legion_We_A  →  Bobiverse_1_We_Are_Legion_We_A  (150×240)
  Bobiverse_2_For_We_Are_Many_-_  →  Bobiverse_2_For_We_Are_Many_-_  (150×150)
  Bobiverse_3_All_These_Worlds_-  →  Bobiverse_3_All_These_Worlds_-  (150×225)
  Bobiverse_4_Heavens_River_-_De  →  Bobiverse_4_Heavens_River_-_De  (150×150)
  Bobiverse_5_Not_Til_We_are_Los  →  Bobiverse_5_Not_Til_We_are_Los  (150×225)
  Cage_of_Souls_2019_cover.jpg    →  Cage_of_Souls_2019_cover.webp   (150×230)
  Consider Phlebas - Iain M. Ban  →  Consider Phleba

## Size Comparison

In [11]:
def fmt(n: int) -> str:
    """Human-readable file size."""
    for unit in ("B", "KB", "MB", "GB"):
        if n < 1024:
            return f"{n:.1f} {unit}"
        n /= 1024
    return f"{n:.1f} TB"

col_w = 30

if results:
    if not col_w:
        col_w = max(len(r[0].name) for r in results)
    header = f"{'File':<{col_w}}   {'Input':>10}   {'Output':>10}   {'Saved':>10}   {'Ratio':>7}"
    print(header)
    print("-" * len(header))

    total_in = total_out = 0
    for src, dst, in_b, out_b in results:
        saved = in_b - out_b
        ratio = out_b / in_b if in_b else 0
        print(f"{src.name[:col_w]:<{col_w}}   {fmt(in_b):>10}   {fmt(out_b):>10}   {fmt(saved):>10}   {ratio:>6.1%}")
        total_in  += in_b
        total_out += out_b

    print("-" * len(header))
    total_saved = total_in - total_out
    total_ratio = total_out / total_in if total_in else 0
    print(f"{'TOTAL':<{col_w}}   {fmt(total_in):>10}   {fmt(total_out):>10}   {fmt(total_saved):>10}   {total_ratio:>6.1%}")

File                                  Input       Output        Saved     Ratio
-------------------------------------------------------------------------------
(Ringworld, Bk 1) Niven, Larry      39.9 KB      36.2 KB       3.7 KB    90.6%
(Ringworld, Bk 2) Niven, Larry      33.3 KB      28.6 KB       4.6 KB    86.1%
(Ringworld, Bk 3) Niven, Larry      52.6 KB      51.1 KB       1.5 KB    97.2%
(Ringworld, Bk 4) Niven, Larry      38.4 KB      33.1 KB       5.2 KB    86.3%
A_Deepness_in_the_Sky_-_Vernor     121.1 KB      35.8 KB      85.3 KB    29.6%
A_Fire_Upon_the_Deep_-_Vernor_     221.6 KB      37.6 KB     183.9 KB    17.0%
Bobiverse_1_We_Are_Legion_We_A      28.3 KB      14.6 KB      13.7 KB    51.5%
Bobiverse_2_For_We_Are_Many_-_      31.9 KB      23.4 KB       8.5 KB    73.3%
Bobiverse_3_All_These_Worlds_-     605.5 KB      25.8 KB     579.6 KB     4.3%
Bobiverse_4_Heavens_River_-_De      48.6 KB      19.6 KB      29.1 KB    40.3%
Bobiverse_5_Not_Til_We_are_Los      94.6 KB      2